# 09 - Evaluate a Trained Model

Run this **after** a training notebook. This is the evaluation notebook: load a saved Mouse Core checkpoint and score it on live environments.

The inference flow is:

1. Build evaluation environments with `EnvConfig` and `make_group_env`.
2. Load a checkpoint with `load_model`.
3. Step environments, convert model predictions to actions with `model.get_action`, and carry the cache returned by `model(..., use_cache=True)`.
4. Record scores and optional render frames.

The rendering code is specific to this example environment; the model loading and cached inference pattern is the reusable Mouse Core part.


In [ ]:
import os

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core import load_model
from mouse_core.models import preferred_dtype
from mouse_core.models.kv_policy import (
    cache_needs_rebuild,
    rebuild_starts,
    resolve_cache_bounds,
)
from mouse_core.data import NumericTokenizer, pack_token_batch

# FrozenLake renders via pygame; run headless in environments without a display.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")


MODEL_ID = "mouse-example-model-offline"      # Hugging Face model repo for load_model
EVAL_STEPS = 512                             # lockstep env steps for inference (passed to run_eval)
TASKS_PER_ENV = 1                             # used only for video score formatting / frame cutoff
MAX_EPISODES_PER_TASK = 20                    # max episodes per task
MAX_STEPS_PER_EPISODE = 30                    # max steps per episode
NUM_ENVS = 10                                 # number of environment streams in the GroupEnv
EVAL_SEED_OFFSET = 1_000_000                  # held-out env seed stream (far from train)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Build Environment

Evaluation uses the same grouped-environment interface as collection and online training. Each config creates one stream, and `make_group_env` lets the notebook step all streams together.

The model was trained on rows containing `action`, `observation`, `reward`, `episode_done`, and `task_done`, so the evaluation environment must emit compatible fields. Rendering is optional and only used here to create replay videos.


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        seed=i + EVAL_SEED_OFFSET,
        episodes_per_task=MAX_EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "render_mode": "rgb_array",
            "max_episode_steps": MAX_STEPS_PER_EPISODE,
            "map_seed": i + EVAL_SEED_OFFSET,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
        },
    )
    for i in range(NUM_ENVS)
]

eval_env = make_group_env(configs)


## Load Model

`load_model` downloads a saved Mouse Core checkpoint and reconstructs the full `Model`, including the embedder, backbone, heads, and weights. You do not need to rebuild the architecture by hand.

On CUDA, this notebook moves the model to bfloat16 for the cached FlexAttention path while keeping output heads stable for action-value prediction. On CPU, inference stays in float32.


In [ ]:
model = load_model(MODEL_ID, force_download=True, map_location="cpu").eval()
model = model.to(device=device, dtype=preferred_dtype(device))

# Packing specs by modality name (strip embedder-only keys like vocab_size/std).
tok_modalities = []
for m in model.encoder.modalities:
    if m.type == "learnable":
        tok_modalities.append({"type": "learnable", "tokens": m.tokens})
        continue
    entry = {"type": m.type, "input_field": m.field}
    if getattr(m, "dim", None) is not None:
        entry["dim"] = m.dim
    tok_modalities.append(entry)

tokenizer = NumericTokenizer(
    input_fields=tok_modalities,
    grouping_field="task_index",
)

transform = tokenizer

def pack_rows(rows: list[list[dict]]):
    steps, sids = [], []
    for i, seq in enumerate(rows):
        for step in seq:
            steps.append(transform(step))
            sids.append(i)
    inputs, _ = pack_token_batch(
        steps,
        sequence_ids=sids if steps else None,
        batch_size=len(rows),
        grouping_field="task_index",
    )
    return inputs


## Batched Incremental Inference With FlexAttention (`run_eval`)

For live control, you usually want to process only the newest rows rather than the full history every step. Call the model with `use_cache=True` and pass the returned cache into the next call:

```python
kv_cache = None
predictions, kv_cache = model(pack_rows(batch), cache=kv_cache, use_cache=True)
actions = model.get_action(predictions, temperature=0.0, num_actions=num_actions)
```

Decode takes model inputs (`TokenBatch`): build them with the same `transform` used at train time (`pack_rows` / `pack_token_batch`). In lockstep environments, each stream can contribute one new step. Ragged per-stream lengths are preserved.

Task boundaries do not clear context or reset KV here — attention isolation comes from the grouping mask. Completed-task scores are read from `env.metrics`.


In [ ]:
eval_contexts = [[] for _ in eval_env.names]

def run_eval(
    *,
    model,
    env,
    contexts: list,
    num_steps: int,
    max_cache: int = 512,
    start_cache=None,
    temperature: float = 0.0,
):
    """Greedy inference with frame capture; scores from ``env.metrics``.

    ``num_steps`` is lockstep ``GroupEnv`` steps for this call.
    Row ``contexts`` persist across the call (mutated in place). The KV cache
    is rebuilt from ``contexts`` at the start (then only on grow/``max_cache``).
    Task boundaries do not clear context or reset KV — attention is task-isolated
    by mask. Completed-task scores come from ``env.metrics``.
    """
    model.eval()
    max_cache, start_cache = resolve_cache_bounds(max_cache, start_cache)
    n = len(env.names)
    if len(contexts) != n:
        raise ValueError(f"contexts has {len(contexts)} streams but env has {n}")
    kv_cache = None
    cached_starts = np.zeros(n, dtype=np.int64)
    cached_ends = np.zeros(n, dtype=np.int64)
    context_start = np.zeros(n, dtype=np.int64)
    for i, c in enumerate(contexts):
        if len(c) > max_cache:
            contexts[i] = c[-max_cache:]
    inputs = None
    outputs = None
    env.metrics.clear()
    num_actions = env.action_space.spaces[0].n
    video_names = env.names
    video_envs = env.envs
    frames_per_env = [[] for _ in video_names]
    steps_done = 0

    def _act_from_contexts(*, rebuild: bool) -> list[dict]:
        nonlocal kv_cache, cached_starts, cached_ends
        ends = np.array([len(c) for c in contexts], dtype=np.int64)
        need_rebuild = rebuild or cache_needs_rebuild(
            has_cache=kv_cache is not None,
            cached_starts=cached_starts,
            cached_ends=cached_ends,
            ends=ends,
            context_start=context_start,
            max_cache=max_cache,
            batch_complete=True,
        )
        with torch.no_grad():
            if need_rebuild:
                starts = rebuild_starts(
                    ends=ends,
                    context_start=context_start,
                    start_cache=start_cache,
                    max_cache=max_cache,
                )
                batch = [contexts[i][int(starts[i]) : int(ends[i])] for i in range(n)]
                predictions, kv_cache = model(pack_rows(batch), use_cache=True)
                cached_starts = starts
                cached_ends = ends.copy()
            else:
                batch = [
                    contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)
                ]
                predictions, kv_cache = model(
                    pack_rows(batch), cache=kv_cache, use_cache=True
                )
                cached_ends = ends.copy()
            actions = model.get_action(
                predictions, temperature=temperature, num_actions=num_actions
            )
        random_inputs = env.sample_random_input()
        return [
            {"action": action} if contexts[i] else random_inputs[i]
            for i, action in enumerate(actions.cpu().numpy())
        ]

    for _ in range(num_steps):
        if outputs is None and any(contexts):
            inputs = _act_from_contexts(rebuild=True)
        elif any(contexts):
            inputs = _act_from_contexts(rebuild=False)
        else:
            inputs = env.sample_random_input()

        outputs = env.step(inputs)
        for i, (inp, out) in enumerate(zip(inputs, outputs)):
            row = {**inp, **out}
            row.pop("info", None)
            contexts[i].append(row)
        for i, video_env in enumerate(video_envs):
            if int(outputs[i]["task_index"]) >= TASKS_PER_ENV:
                continue
            frames = video_env.render()
            if frames:
                frames_per_env[i].append(frames[-1].copy())
        steps_done += 1
        if steps_done % 100 == 0:
            task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
            mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
            print(
                f"step {steps_done} | {len(task_scores)} tasks this window | "
                f"mean task score {mean_task_score:.2f}/{MAX_EPISODES_PER_TASK}"
            )

    scores_per_env = [list(scores) for scores in env.metrics.task_cum_rewards]
    return video_names, frames_per_env, scores_per_env


video_names, frames_per_env, eval_task_scores = run_eval(
    model=model, env=eval_env, contexts=eval_contexts, num_steps=EVAL_STEPS
)
eval_env.close()


## Replay MP4s

The remaining cells format environment-specific render frames as MP4s. This is not required for Mouse Core inference, but it is a useful way to inspect what the loaded policy is doing.


In [ ]:
def format_task_scores(*, task_scores):
    task_scores = task_scores[:TASKS_PER_ENV]
    scores_text = ' '.join((f'{score:.0f}' for score in task_scores))
    return f'task scores [{scores_text}] / {MAX_EPISODES_PER_TASK} | mean {sum(task_scores) / TASKS_PER_ENV:.2f}'

def display_env_replay(*, name, frames, task_scores, fps=5):
    print(f'{name}: {format_task_scores(task_scores=task_scores)}')
    if not frames:
        print('  (no frames captured)')
        return
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(w / 100, h / 100))
    ax.axis('off')
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)
    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps, blit=True)
    plt.close(fig)
    display(HTML(ani.to_html5_video()))
for name, frames, task_scores in zip(video_names, frames_per_env, eval_task_scores):
    display_env_replay(name=name, frames=frames, task_scores=task_scores)
